# Ejercicio 1: línea base, un árbol y un bosque

`viajes_features.csv`, el mismo de la sesión pasada. La preparación es tu pipeline de la Semana 9, Sesión 2, y no está copiada aquí: vive en `preparacion_taxis.py`, en esta misma carpeta, y el notebook la importa. Así todos los modelos usan la misma preparación y la misma división.

Corre todas las celdas antes de tocar nada (**Run All**, en la barra de arriba del notebook): corre de principio a fin sin los `TODO` resueltos. Después resuelve cada `TODO` y vuelve a correr solo la celda de ese modelo y la de la tabla: los demás modelos no se vuelven a entrenar.

In [ ]:
import time

import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from preparacion_taxis import dividir, preparacion

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

## La función que evalúa cada modelo

In [ ]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    """Entrena el modelo despues de la preparacion y mide el error absoluto
    medio en entrenamiento y en prueba. Guarda la fila de la tabla en
    resultados y el pipeline entrenado en entrenados, los dos con su
    nombre: si vuelves a correr la celda de un modelo, se reemplazan, no
    quedan repetidos."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    resultados[nombre] = {
        "modelo": nombre,
        "mae_entrenamiento": round(mae_ent, 4),
        "mae_prueba": round(mae_prueba, 4),
        "brecha": round(mae_prueba - mae_ent, 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:28s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}   {segundos:6.2f} s")

## Paso 1: línea base

In [ ]:
# TODO 1: la linea base, un DummyRegressor que siempre predice la mediana
# de la tarifa de entrenamiento.
linea_base = DummyRegressor()
evaluar("linea base", linea_base)

## Paso 2: un árbol sin límite

In [ ]:
# TODO 2: un arbol de decision (DecisionTreeRegressor) sin ningun limite,
# con random_state=42.
arbol = DummyRegressor()
evaluar("arbol sin limite", arbol)

## Paso 3: el mismo árbol, regularizado

In [ ]:
# TODO 3: el mismo arbol, regularizado con max_depth=8.
arbol_limitado = DummyRegressor()
evaluar("arbol max_depth=8", arbol_limitado)

## Paso 4: un bosque aleatorio

In [ ]:
# TODO 4: un bosque aleatorio (RandomForestRegressor) de 200 arboles, con
# random_state=42.
bosque = DummyRegressor()
evaluar("bosque aleatorio", bosque)

## Tabla de resultados

In [ ]:
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("resultados_ejercicio1.csv", index=False)
tabla